# Data Wrangling — Synthetic Vietnamese Students’ Feedback

Nguồn: [Kaggle — toreleon/synthetic-vietnamese-students-feedback-corpus](https://www.kaggle.com/datasets/toreleon/synthetic-vietnamese-students-feedback-corpus)

1. `student_feedback_data/synthetic_train.csv` (train)  
2. `student_feedback_data/synthetic_val.csv` (validation)  

Dữ liệu **văn bản**: missing → làm sạch câu; chuẩn hoá đơn vị → điểm sentiment; normalize → độ dài câu; binning → ngắn/trung/dài; dummy → `topic`.

Mỗi bước có **1 dòng mẫu**. Dòng `# TODO` em làm tương tự (đổi tên cột / file).

## Download data and explore

Luôn bắt đầu bằng **nhìn dữ liệu**, đừng nhảy vào `fillna`. Hỏi: bao nhiêu dòng? cột nào object? câu rỗng / chỉ khoảng trắng?

| Cột | Kiểu | Ý nghĩa |
|---|---|---|
| `sentence` | object | Câu phản hồi (VN hoặc EN). Không trùng trong từng file. |
| `sentiment` | object | Nhãn cảm xúc: `negative` / `neutral` / `positive` (gần cân bằng). |
| `topic` | object | Chủ đề: `lecturer`, `curriculum`, `facility`, `others`. |

**Ghi chú:** Không có `NaN` sẵn. Một số câu là tiếng Anh (không dấu). Cần tạo biến số (độ dài câu) để normalize / binning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

path_train = "student_feedback_data/synthetic_train.csv"
path_val = "student_feedback_data/synthetic_val.csv"

df_train = pd.read_csv(path_train)  # mẫu: đọc train
# TODO: đọc path_val vào df_val (giống dòng trên)
df_val = pd.read_csv(path_val)
df_val = None

print("Train:", df_train.shape)  # mẫu: in số dòng, số cột
# TODO: in shape của df_val
print("Val:", df_val.shape)
print(df_train.dtypes)  # mẫu: xem kiểu cột
df_train.head()  # mẫu: 5 dòng đầu
# TODO: xem 5 dòng đầu df_val
df_val.head()

---
# Bộ 1 — Train

## 1. Xử lý giá trị thiếu (Missing Values)

Câu rỗng / chỉ khoảng trắng → `NaN`. `sentiment` / `topic` (phân loại) → **mode**. `sentence` là nội dung chính → **drop** dòng thiếu rồi `reset_index`.

In [ ]:
df = df_train.copy()  # mẫu: làm việc trên bản sao

df["sentence"] = df["sentence"].astype(str).str.strip()  # mẫu: cắt khoảng trắng câu
# TODO: strip tương tự cho sentiment và topic

# TODO: replace {"nan": np.nan, "": np.nan, "None": np.nan} trên 3 cột (có thể dùng vòng for)
cols = ["sentence", "sentiment", "topic"]
for c in cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].replace({"nan": np.nan, "": np.nan, "None": np.nan})
print(df.isnull().sum())  # mẫu: đếm missing trước xử lý

df["sentiment"] = df["sentiment"].fillna(df["sentiment"].mode(dropna=True)[0])  # mẫu: cột chữ → mode
# TODO: fillna mode cho cột topic (copy mẫu, đổi tên cột)
df["topic"] = df["topic"].fillna(df["topic"].mode(dropna=True)[0])
# TODO: dropna subset=["sentence"] rồi reset_index (câu là nội dung chính — không điền mode)
df = df.dropna(subset=["sentence"]).reset_index(drop=True)
# TODO: in lại isnull().sum() và len(df)
print(df.isnull().sum())
print("Tổng số dòng còn lại:", len(df))

## 2. Sửa định dạng dữ liệu (Correct Data Format)

Ép `sentiment`, `topic` về kiểu `category`. Thêm độ dài câu (`n_chars`, `n_words`) dạng số.

In [ ]:
print(df.dtypes)  # mẫu: kiểu trước khi ép

df["sentiment"] = df["sentiment"].astype("category")  # mẫu: chữ → category
# TODO: ép topic về category (giống dòng trên)
df["topic"] = df["topic"].astype("category")
df["n_chars"] = df["sentence"].str.len().astype(int)  # mẫu: số ký tự
# TODO: n_words = sentence.str.split().str.len() rồi astype(int)
df["n_words"] = df["sentence"].str.split().str.len().astype(int)
df[["sentence", "n_chars"]].head()  # mẫu
# TODO: head thêm cột n_words; in lại dtypes

df[["sentence", "n_chars", "n_words"]].head()
print(df.dtypes)


## 3. Chuẩn hoá dữ liệu (Data Standardization)

Đưa nhãn cảm xúc về **cùng thang số**: `negative → -1`, `neutral → 0`, `positive → 1`.

In [ ]:
sentiment_map = {"negative": -1, "neutral": 0, "positive": 1}  # mẫu: từ điển nhãn → số

# TODO: cột sentiment_score = df["sentiment"].map(sentiment_map).astype(int)
df["sentiment_score"] = df["sentiment"].map(sentiment_map).astype(int)
df[["sentiment"]].drop_duplicates()  # mẫu: xem các nhãn
# TODO: head / drop_duplicates hai cột sentiment và sentiment_score
df[["sentiment", "sentiment_score"]].drop_duplicates()

## 4. Chuẩn hoá phạm vi giá trị (Data Normalization)

`n_chars` lớn hơn `n_words` → `x / x.max()` về 0–1.

In [ ]:
df["n_chars_normalized"] = df["n_chars"] / df["n_chars"].max()  # mẫu: chia max → 0–1
# TODO: n_words_normalized = n_words / n_words.max()
df["n_words_normalized"] = df["n_words"] / df["n_words"].max()
df[["n_chars", "n_chars_normalized"]].head()  # mẫu
# TODO: head thêm n_words và n_words_normalized
df[["n_chars", "n_chars_normalized", "n_words", "n_words_normalized"]].head()

## 5. Phân nhóm (Binning)

Chia độ dài câu (số từ) thành Short / Medium / Long bằng `pd.cut()`.

In [ ]:
bins = np.linspace(df["n_words"].min(), df["n_words"].max(), 4)  # mẫu: 3 khoảng đều

df["length_binned"] = pd.cut(
    df["n_words"], bins=bins, labels=["Short", "Medium", "Long"], include_lowest=True
)  # mẫu: gán nhãn

print(df["length_binned"].value_counts())  # mẫu: đếm mỗi nhóm

# TODO: plt.hist(df["n_words"], bins=3) rồi xlabel / ylabel / title / show
print(df["length_binned"].value_counts())

plt.figure(figsize=(7, 4))
plt.hist(df["n_words"], bins=3, edgecolor="black", color="skyblue")
plt.xlabel("Số từ (n_words)")
plt.ylabel("Tần suất (frequency)")
plt.title("Phân phối độ dài câu theo nhóm (Train)")
plt.show()
df[["n_words", "length_binned"]].head()

## 6. Biến chỉ thị / Biến giả (Dummy Variable)

`topic` (chữ) → mỗi chủ đề một cột 0/1. Giữ `sentiment` làm nhãn dự đoán.

In [ ]:
dummy_topic = pd.get_dummies(df["topic"], prefix="topic", dtype=int)  # mẫu: cột 0/1

# TODO: gộp dummy vào df — pd.concat([df, dummy_topic], axis=1)
df = pd.concat([df, dummy_topic], axis=1)

# TODO: xoá cột gốc topic — df.drop(columns=["topic"])  (giữ sentiment)
df = df.drop(columns=["topic"])
df.head()  # mẫu

## Check and save

Trước khi `to_csv`:

* `isnull().sum()` còn 0 (hoặc đúng chỗ cố ý để thiếu)
* `n_chars`, `n_words` là số
* có cột mới: `sentiment_score`, `*_normalized`, `length_binned`, `topic_*`

`index=False` để file không thêm cột index 0,1,2,…


In [ ]:
df.to_csv("student_feedback_train_clean.csv", index=False)  # mẫu: lưu train
print(df.shape)

# TODO: (tuỳ chọn) gán df_train_clean = df
df_train_clean = df.copy()

---
---
# Bộ 2 — Validation

Làm **giống bộ train**, đổi `df_train` → `df_val`. Copy mẫu ở trên, sửa tên biến / title. Kết thúc bằng **Check and save** (`to_csv`, `index=False`).


In [ ]:
df = df_val.copy()  # mẫu: bắt đầu từ validation

df["sentence"] = df["sentence"].astype(str).str.strip()  # mẫu bước 1
# TODO: strip + replace nan/""/"None" cho sentiment, topic; fillna mode; dropna sentence
cols = ["sentence", "sentiment", "topic"]
for c in cols:
    df[c] = df[c].astype(str).str.strip()
    df[c] = df[c].replace({"nan": np.nan, "": np.nan, "None": np.nan})
    
    df["sentiment"] = df["sentiment"].fillna(df["sentiment"].mode(dropna=True)[0])
df["topic"] = df["topic"].fillna(df["topic"].mode(dropna=True)[0])
df = df.dropna(subset=["sentence"]).reset_index(drop=True)

df["sentiment"] = df["sentiment"].astype("category")  # mẫu bước 2
# TODO: ép topic; tạo n_chars (đã có mẫu train) và n_words
df["topic"] = df["topic"].astype("category")
df["n_chars"] = df["sentence"].str.len().astype(int)
df["n_words"] = df["sentence"].str.split().str.len().astype(int)


sentiment_map = {"negative": -1, "neutral": 0, "positive": 1}  # mẫu bước 3
# TODO: sentiment_score = map rồi astype(int)
df["sentiment_score"] = df["sentiment"].map(sentiment_map).astype(int)


df["n_chars_normalized"] = df["n_chars"] / df["n_chars"].max()  # mẫu bước 4
# TODO: n_words_normalized
df["n_words_normalized"] = df["n_words"] / df["n_words"].max()


# TODO: bước 5 — linspace + pd.cut + value_counts + hist (title Val)
bins = np.linspace(df["n_words"].min(), df["n_words"].max(), 4)
df["length_binned"] = pd.cut(
    df["n_words"], bins=bins, labels=["Short", "Medium", "Long"], include_lowest=True
)

print(df["length_binned"].value_counts())

plt.figure(figsize=(7, 4))
plt.hist(df["n_words"], bins=3, edgecolor="black", color="salmon")
plt.xlabel("Số từ (n_words)")
plt.ylabel("Tần suất (frequency)")
plt.title("Phân phối độ dài câu theo nhóm (Validation)")
plt.show()

dummy_topic = pd.get_dummies(df["topic"], prefix="topic", dtype=int)  # mẫu bước 6
# TODO: concat + drop topic
df = pd.concat([df, dummy_topic], axis=1)
df = df.drop(columns=["topic"])
# TODO: to_csv student_feedback_val_clean.csv; print shape; df.head()
df.to_csv("student_feedback_val_clean.csv", index=False)
print(df.shape)